# X5_Q1 — Tabla maestra + variables de estado del activo

Primer notebook de **X5_alt_v2** (evolución paralela de X5_alt — ver
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10,
documento rector [`docs/plans/x5_contexto_conversacion.md`](../docs/plans/x5_contexto_conversacion.md)).
No reemplaza `X5_P1.ipynb`/`X5_P2.ipynb` (v1, congelados en `scripts/otros_strings/`).

Este notebook construye, para cada activo de `valores`, la misma tabla
continua en el tiempo (una fila por vela H1) que ya resolvía `X5_P1.ipynb` —
precio + X3 (técnicos) + X2 (fundamental) — y la deja como punto de partida
para las variables de estado del activo (drawdown, volatilidad, ruido,
tendencia/régimen, liquidez) que se agregan en los ítems siguientes de
`X5_alt_v2` (TODOs en `docs/tracking/todos.md`, sección X5_alt_v2).

**Input único que se puede cambiar — lista, no string** (a diferencia del
`valor` singular de v1): soporta varios activos sin reescribir esta celda.

In [ ]:
import sys
from pathlib import Path

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402

valores = list(cfg.VALORES)  # default = todos los activos de config.py; recortar a mano acá para un análisis puntual con menos

## 0. Preparación

Reutilizamos `_calcular_todos_indicadores` de `X3_technical_features.py`
(igual que `X5_P1.ipynb`) en vez de reimplementar los indicadores técnicos.

In [ ]:
import json  # para leer x2_history.json

import numpy as np
import pandas as pd

from X3_technical_features import _calcular_todos_indicadores, _wilder_ema  # (reusa X3 en vez de reimplementar indicadores)

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

print(f"Activos: {valores}")

## 1. Tabla base por activo (precio + X3 + X2)

Misma lógica de ensamblado que `X5_P1.ipynb` (precio OHLCV + indicadores
técnicos de X3 + fundamental de X2 con forward-fill), encapsulada en una
función para poder correrla sobre cada activo de `valores`.

In [ ]:
def _cargar_tabla_base(valor):
    # Precio: el CSV no viene ordenado por fecha y puede traer velas duplicadas (ver X5_P1 §1)
    path_precio = cfg.CARPETA_DATA / f'{valor}.csv'
    df_precio = pd.read_csv(path_precio)
    df_precio['DateTime'] = pd.to_datetime(df_precio['DateTime'])
    df_precio = (df_precio.sort_values('DateTime')
                           .drop_duplicates(subset=['DateTime'])
                           .reset_index(drop=True))

    # X3 sobre el histórico completo (no incremental) para que indicadores con
    # warm-up largo (EMA, RSI, SMA 200) sean correctos desde la primera vela útil.
    # conjunto_N vacío = sin distancia a soportes (fuera de v0, igual que X5_P1 §2).
    df_x3 = _calcular_todos_indicadores(df_precio, set())
    df_x3 = df_x3.drop(columns=['dist_nearest_support', 'dist_floor_support', 'density_2pct'])
    df_x3 = df_x3.rename(columns={c: f'x3_{c}' for c in df_x3.columns if c != 'datetime'})

    # X2: x2_history.json trae todos los activos mezclados — filtramos por {valor}
    path_x2 = cfg.CARPETA_FUNDAMENTALS / 'x2_history.json'
    with open(path_x2) as f:
        x2_hist = json.load(f)
    registros = [e for e in x2_hist if e.get('activo') == valor]
    cols_x2 = ['DateTime', 'x2_score', 'x2_score_cross', 'x2_score_tendencia']
    df_x2 = pd.DataFrame([
        {
            'DateTime': pd.to_datetime(e.get('periodo_inicio') or e['date']),
            'x2_score': e.get('score'),
            'x2_score_cross': e.get('score_cross'),
            'x2_score_tendencia': e.get('score_tendencia'),
        }
        for e in registros
    ], columns=cols_x2)
    # columns= evita que un activo sin registros aún (ej. MSFT recién agregado a VALORES) quede con
    # un DataFrame sin columnas; el cast explícito evita el MergeError de merge_asof por dtype
    # object (DateTime vacío) vs. datetime64 cuando no hay filas para inferir el tipo.
    df_x2['DateTime'] = pd.to_datetime(df_x2['DateTime'])
    df_x2 = df_x2.sort_values('DateTime').reset_index(drop=True)  # merge_asof exige orden ascendente

    # Precio y X3 comparten índice/orden → join exacto por DateTime
    tabla = df_precio.merge(df_x3, left_on='DateTime', right_on='datetime').drop(columns=['datetime'])
    # merge_asof(direction='backward') = forward-fill del último score de X2 conocido
    tabla = pd.merge_asof(tabla, df_x2, on='DateTime', direction='backward')
    return tabla


tablas = {valor: _cargar_tabla_base(valor) for valor in valores}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas — {tabla['DateTime'].min()} → {tabla['DateTime'].max()}")

## 2. Técnicos filtrados y transformados (Q1-3)

`X3_technical_features.py` ya calcula niveles crudos de medias móviles
(`x3_sma_{20,50,200}`, `x3_ema_{12,26}`) — triviales como variable de estado
porque viven en la escala del propio precio (correlación ~1 con `Close` por
construcción, no aportan señal nueva). Se descartan esos niveles crudos y se
reemplazan por dos transformaciones normalizadas por ATR (independientes de
la escala de precio y del régimen de volatilidad, a diferencia de
`x3_sma_dist_{w}`/`x3_ema_dist_{w}` que ya existen normalizados por `Close`):

- `distancia_sma_{w}` / `distancia_ema_{w}` = `(Close - media_{w}) / ATR_14`
- `pendiente_sma_{w}` / `pendiente_ema_{w}` = cambio promedio por período de
  la media móvil en los últimos `w` períodos, normalizado por `ATR_14`

In [4]:
def _filtrar_y_transformar_tecnicos(df):
    atr = df['x3_atr_14']
    for w in cfg.X3_VENTANAS['sma']:
        col = f'x3_sma_{w}'
        df[f'distancia_sma_{w}'] = (df['Close'] - df[col]) / atr
        df[f'pendiente_sma_{w}'] = (df[col].diff(w) / w) / atr
    for w in cfg.X3_VENTANAS['ema']:
        col = f'x3_ema_{w}'
        df[f'distancia_ema_{w}'] = (df['Close'] - df[col]) / atr
        df[f'pendiente_ema_{w}'] = (df[col].diff(w) / w) / atr

    # niveles crudos descartados (trivial vs. Close); se conservan x3_sma_dist_*/x3_ema_dist_* (ya normalizados por Close)
    cols_crudas = [f'x3_sma_{w}' for w in cfg.X3_VENTANAS['sma']] + [f'x3_ema_{w}' for w in cfg.X3_VENTANAS['ema']]
    return df.drop(columns=cols_crudas)


tablas = {valor: _filtrar_y_transformar_tecnicos(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    nuevas = [c for c in tabla.columns if c.startswith(('distancia_', 'pendiente_'))]
    print(f"{valor}: {tabla.shape[1]} columnas — agregadas {nuevas}")

BTCUSD: 42 columnas — agregadas ['distancia_sma_20', 'pendiente_sma_20', 'distancia_sma_50', 'pendiente_sma_50', 'distancia_sma_200', 'pendiente_sma_200', 'distancia_ema_12', 'pendiente_ema_12', 'distancia_ema_26', 'pendiente_ema_26']


## 3. Variables de drawdown del activo (Q1-4)

Drawdown **del activo** (nunca de cuenta/posiciones — eso es responsabilidad
de X1/X4). Dos vistas complementarias:

- **Histórico**: `drawdown_historico` contra el máximo histórico acumulado
  (`cummax`), más `duracion_drawdown` (períodos desde el último máximo),
  `velocidad_caida` (profundidad / duración: caída promedio por período desde
  el máximo) y `recuperacion_desde_minimo` (cuánto se recuperó el precio desde
  el mínimo del episodio de drawdown actual — un episodio nuevo empieza en
  cada máximo histórico).
- **Por ventana**: `drawdown_7d/30d/90d/365d` contra el máximo rodante de esa
  ventana — a diferencia del histórico, "resetea" la referencia si el precio
  alcanzó un máximo reciente aunque no sea el máximo histórico absoluto.

In [5]:
def _variables_drawdown(df):
    close = df['Close']
    cummax = close.cummax()
    df['drawdown_historico'] = (close - cummax) / cummax

    nuevo_maximo = close >= cummax
    episodio = nuevo_maximo.cumsum()
    df['duracion_drawdown'] = close.groupby(episodio).cumcount()
    min_episodio = close.groupby(episodio).cummin()
    df['recuperacion_desde_minimo'] = (close - min_episodio) / min_episodio
    df['velocidad_caida'] = df['drawdown_historico'] / df['duracion_drawdown'].replace(0, np.nan)

    for dias, horas in [(7, 168), (30, 720), (90, 2160), (365, 8760)]:
        rolling_max = close.rolling(horas, min_periods=1).max()
        df[f'drawdown_{dias}d'] = (close - rolling_max) / rolling_max

    return df


tablas = {valor: _variables_drawdown(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[1]} columnas — drawdown_historico en [{tabla['drawdown_historico'].min():.3f}, {tabla['drawdown_historico'].max():.3f}], "
          f"duracion_drawdown max {tabla['duracion_drawdown'].max()} períodos")

BTCUSD: 50 columnas — drawdown_historico en [-0.772, 0.000], duracion_drawdown max 20296 períodos


## 4. Variables de volatilidad (Q1-5)

ATR absoluto/relativo y volatilidad realizada corta/larga ya los calcula X3
(`x3_atr_14`, `x3_atr_pct`, `x3_vol_24h`, `x3_vol_7d`) — se reutilizan tal
cual, sin reimplementar. Se agregan 3 variables nuevas, todas sobre retornos
logarítmicos horarios:

- `cambio_volatilidad` = `vol_24h / vol_7d` — >1 indica un salto de
  volatilidad reciente sobre la línea base de 7 días.
- `vol_alcista_30d` / `vol_bajista_30d` = desviación estándar anualizada de
  solo los retornos positivos / negativos en una ventana de 30 días — separa
  el riesgo al alza del riesgo a la baja (una serie casi simétrica en retorno
  medio puede no serlo en volatilidad direccional).
- `frecuencia_extremos_30d` = fracción de horas, en los últimos 30 días, con
  `|retorno log| > 2 × desviación estándar expandida hasta ese punto` — el
  umbral usa `expanding()` (no rolling) para no mirar información futura al
  definirlo; proxy de cuán seguido el activo tiene movimientos atípicos.

In [6]:
def _variables_volatilidad(df):
    close = df['Close']
    log_ret = np.log(close / close.shift(1))

    df['cambio_volatilidad'] = df['x3_vol_24h'] / df['x3_vol_7d']

    ret_up = log_ret.where(log_ret > 0)
    ret_down = log_ret.where(log_ret < 0)
    df['vol_alcista_30d'] = ret_up.rolling(720, min_periods=50).std(ddof=1) * np.sqrt(8760)
    df['vol_bajista_30d'] = ret_down.rolling(720, min_periods=50).std(ddof=1) * np.sqrt(8760)

    umbral = 2 * log_ret.expanding(min_periods=100).std(ddof=1)
    extremo = log_ret.abs() > umbral
    df['frecuencia_extremos_30d'] = extremo.rolling(720, min_periods=50).mean()

    return df


tablas = {valor: _variables_volatilidad(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[1]} columnas — cambio_volatilidad media {tabla['cambio_volatilidad'].mean():.3f}, "
          f"frecuencia_extremos_30d media {tabla['frecuencia_extremos_30d'].mean():.3f}")

BTCUSD: 54 columnas — cambio_volatilidad media 0.938, frecuencia_extremos_30d media 0.039


## 5. Variables de liquidez (Q1-8)

`Spread` y `Tick_Volume` vienen directo del CSV de MT5 (`Real_Volume` siempre
en 0 — broker no lo reporta, ver `CLAUDE.md`). Slippage no se agrega: no
existe en los datos de `Data/` — es una métrica de ejecución (brecha OE→OA)
que vive en los logs de X1/X4, no en la tabla de estado del activo.

- `spread_relativo` = `Spread / media_móvil_30d(Spread)` — `Spread` viene en
  puntos del bróker MT5 (requiere `symbol_info(valor).point` para pasarlo a
  precio, dato no disponible en Mac sin MT5 ni persistido en los CSVs), así
  que no se puede dividir por `Close` sin asumir un factor de conversión no
  verificable. Normalizar contra su propio promedio móvil evita ese supuesto:
  >1 indica spread más ancho que lo habitual reciente para ese activo.
- `tick_volume_z` = z-score de `Tick_Volume` contra media/desviación móviles
  de 30 días — normaliza el volumen frente a la tendencia de crecimiento de
  actividad del activo en el tiempo, no solo frente al nivel reciente.
- `volumen_relativo` = `Tick_Volume / media_móvil_30d(Tick_Volume)` — volumen
  relativo (RVOL): >1 indica actividad por sobre su propio promedio reciente,
  útil para detectar picos puntuales que el z-score (más sensible a la
  dispersión) puede diluir.

In [7]:
def _variables_liquidez(df):
    spread = df['Spread']
    media_spread_30d = spread.rolling(720, min_periods=50).mean()
    df['spread_relativo'] = spread / media_spread_30d

    vol = df['Tick_Volume']
    media_30d = vol.rolling(720, min_periods=50).mean()
    std_30d = vol.rolling(720, min_periods=50).std(ddof=1)
    df['tick_volume_z'] = (vol - media_30d) / std_30d
    df['volumen_relativo'] = vol / media_30d

    return df


tablas = {valor: _variables_liquidez(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[1]} columnas — spread_relativo media {tabla['spread_relativo'].mean():.3f}, "
          f"volumen_relativo media {tabla['volumen_relativo'].mean():.3f}")

BTCUSD: 57 columnas — spread_relativo media 0.999, volumen_relativo media 1.005


## 6. Variables fundamentales de estado (Q1-10)

Las columnas `x2_score`/`x2_score_cross`/`x2_score_tendencia` ya están en la
tabla desde la Sección 1 (`merge_asof` backward = forward-fill del último
valor conocido). `x2_history.json` es muy escaso para BTCUSD (solo 2 registros
en todo el horizonte, ver `X5_alternativo.md` §2) — casi todo el historial
queda con el mismo score forward-filled. Las variables de esta sección
existen justamente para distinguir una actualización real de la mera
repetición del forward-fill:

- `fundamental_cambio` = `x2_score.diff()` — 0 en la inmensa mayoría de las
  filas (forward-fill), distinto de 0 solo donde llega una actualización real.
- `fundamental_direccion_cambio` = signo de `fundamental_cambio` (`+1`/`-1`/`0`).
- `fundamental_magnitud_cambio` = `|fundamental_cambio|`.
- `fundamental_edad` = velas H1 desde la última actualización real (mismo
  patrón de `groupby().cumcount()` que `duracion_drawdown` en la Sección 3);
  mide qué tan vieja es la información vigente.
- `fundamental_frecuencia_efectiva_90d` = fracción de velas, en una ventana
  rodante de 90 días (2160h — más ancha que los 30d del resto del notebook
  para tener chance real de capturar alguna actualización dado lo escaso de
  `x2_history`), que corresponden a una actualización real.
- `fundamental_estabilidad_90d` = desviación estándar rodante de `x2_score`
  en la misma ventana de 90 días — con forward-fill queda en 0 casi siempre y
  solo se mueve alrededor de una actualización; complementa a la frecuencia
  (cuenta eventos) con la magnitud del cambio.
- `fundamental_contradiccion` = `(x2_score_cross - 0.5) * (x2_score_tendencia - 0.5)`
  — ambos componentes están centrados en 0.5 (ver `X2_fundamentals.py`):
  negativo = contradicción (un componente por sobre neutro y el otro por
  debajo), positivo = consistencia, 0 = al menos uno es neutro.

In [8]:
def _variables_fundamentales(df):
    score = df['x2_score']
    cambio = score.diff()
    df['fundamental_cambio'] = cambio
    df['fundamental_direccion_cambio'] = np.sign(cambio)
    df['fundamental_magnitud_cambio'] = cambio.abs()

    actualizo = cambio.fillna(0) != 0
    grupo = actualizo.cumsum()
    df['fundamental_edad'] = df.groupby(grupo).cumcount()

    df['fundamental_frecuencia_efectiva_90d'] = actualizo.rolling(2160, min_periods=50).mean()
    df['fundamental_estabilidad_90d'] = score.rolling(2160, min_periods=50).std(ddof=1)

    df['fundamental_contradiccion'] = (df['x2_score_cross'] - 0.5) * (df['x2_score_tendencia'] - 0.5)

    return df


tablas = {valor: _variables_fundamentales(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    n_actualizaciones = int((tabla['fundamental_cambio'].fillna(0) != 0).sum())
    print(f"{valor}: {tabla.shape[1]} columnas — {n_actualizaciones} actualizaciones reales de x2_score, "
          f"fundamental_edad max {tabla['fundamental_edad'].max()} períodos, "
          f"fundamental_contradiccion media {tabla['fundamental_contradiccion'].mean():.4f}")

BTCUSD: 64 columnas — 0 actualizaciones reales de x2_score, fundamental_edad max 40282 períodos, fundamental_contradiccion media nan


## 7. Variables de ruido (Q1-6)

Todas sobre retornos logarítmicos horarios y en ventana de 30 días (720h,
igual que el resto de métricas de 30d del notebook), reutilizando
`x3_atr_pct` de X3 en vez de recalcular ATR:

- `ruido_cambios_signo_30d` = fracción de velas, en la ventana, donde el
  signo del retorno log cambia respecto a la vela anterior — proxy de
  zigzag/whipsaw frente a movimiento direccional sostenido.
- `ruido_ratio_movimiento_30d` = `suma(|retorno log|)` en la ventana /
  `|retorno log acumulado|` en la misma ventana — inverso del Efficiency
  Ratio de Kaufman: cercano a 1 = recorrido ≈ desplazamiento neto (movimiento
  eficiente), alto = mucho recorrido para poco desplazamiento neto (ruido).
- `ruido_mecha_30d` = promedio rodante de `(High-Low-|Close-Open|)/(High-Low)`
  por vela — mismo concepto que la fuerza del rechazo de X0
  (`f = 1-|Close-Open|/(High-Low)`, ver `CLAUDE.md` Paso 2) pero agregado en
  ventana como variable de estado en vez de por vela; alto = velas con mecha
  larga en relación a su rango, indecisión reciente.
- `ruido_atr_sin_direccion_30d` = `x3_atr_pct` promedio en la ventana /
  `|retorno log acumulado|` en la misma ventana — ATR alto con retorno
  acumulado bajo; complementa a `ruido_ratio_movimiento_30d` midiendo lo mismo
  con el rango típico suavizado (ATR) en vez de la suma vela a vela de
  retornos.

In [9]:
def _variables_ruido(df):
    close = df['Close']
    log_ret = np.log(close / close.shift(1))

    signo = np.sign(log_ret)
    df['ruido_cambios_signo_30d'] = (signo != signo.shift(1)).rolling(720, min_periods=50).mean()

    retorno_30d = np.log(close / close.shift(720))
    movimiento_total_30d = log_ret.abs().rolling(720, min_periods=50).sum()
    df['ruido_ratio_movimiento_30d'] = movimiento_total_30d / retorno_30d.abs().replace(0, np.nan)

    rango = df['High'] - df['Low']
    mecha = (rango - (df['Close'] - df['Open']).abs()) / rango.replace(0, np.nan)
    df['ruido_mecha_30d'] = mecha.rolling(720, min_periods=50).mean()

    df['ruido_atr_sin_direccion_30d'] = df['x3_atr_pct'].rolling(720, min_periods=50).mean() / retorno_30d.abs().replace(0, np.nan)

    return df


tablas = {valor: _variables_ruido(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[1]} columnas — ruido_cambios_signo_30d media {tabla['ruido_cambios_signo_30d'].mean():.3f}, "
          f"ruido_mecha_30d media {tabla['ruido_mecha_30d'].mean():.3f}")

BTCUSD: 68 columnas — ruido_cambios_signo_30d media 0.529, ruido_mecha_30d media 0.572


## 8. Variables de tendencia/régimen (Q1-7)

- `adx_14` — Average Directional Index (ADX) de Wilder, ventana 14 (misma
  ventana que `x3_atr_14`). Mide **fuerza** de la tendencia sin importar
  dirección: valores bajos (< 20, convención estándar de Wilder) indican
  mercado sin tendencia clara (lateral), valores altos indican tendencia
  fuerte (alcista o bajista). Reutiliza `_wilder_ema` de
  `X3_technical_features.py` (mismo suavizado que usa `_calcular_atr`) en vez
  de reimplementar un esquema de suavizado distinto.
- `regimen_tendencia` — categórica (`'bull'` / `'bear'` / `'sideways'`,
  terminología usada tal cual en `x5_contexto_conversacion.md` §21):
  `'sideways'` si `adx_14 < 20` (tendencia débil, independiente del signo de
  la pendiente); si no, `'bull'`/`'bear'` según el signo de
  `x3_trend_slope_50` (pendiente OLS normalizada ya calculada por X3, no se
  reimplementa). `NaN` durante el warm-up de `adx_14`/`x3_trend_slope_50`
  para no clasificar filas sin suficiente historia.
- `ruptura_rango` — ruptura de rango (estilo canal de Donchian) contra la
  ventana corta de 20 períodos ya usada por X3 para `drawdown`/`trend_slope`
  (`X3_VENTANAS['drawdown'][0]`): `+1` si `Close` supera el máximo de las 20
  velas previas (ruptura al alza), `-1` si perfora el mínimo (ruptura a la
  baja), `0` si permanece dentro del rango. `NaN` en el warm-up.
- `duracion_en_rango` — períodos transcurridos desde la última ruptura
  (mismo patrón `groupby().cumcount()` que `duracion_drawdown` en la
  Sección 3): alto = el precio lleva mucho tiempo sin romper el rango
  reciente (permanencia), bajo = ruptura reciente.

In [10]:
def _variables_tendencia_regimen(df):
    high, low, close = df['High'], df['Low'], df['Close']

    # ADX de Wilder: +DM/-DM y TR suavizados igual que _calcular_atr (mismo _wilder_ema)
    up_move = high.diff()
    down_move = -low.diff()
    plus_dm = up_move.where((up_move > down_move) & (up_move > 0), 0.0)
    minus_dm = down_move.where((down_move > up_move) & (down_move > 0), 0.0)
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)

    w_adx = cfg.X3_VENTANAS['atr']
    tr_suavizado = _wilder_ema(tr, w_adx)
    plus_di = 100 * _wilder_ema(plus_dm, w_adx) / tr_suavizado
    minus_di = 100 * _wilder_ema(minus_dm, w_adx) / tr_suavizado
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
    df['adx_14'] = _wilder_ema(dx, w_adx)

    pendiente = df['x3_trend_slope_50']
    sin_historia = df['adx_14'].isna() | pendiente.isna()
    regimen = np.select(
        [df['adx_14'] < 20, pendiente > 0, pendiente < 0],
        ['sideways', 'bull', 'bear'],
        default='sideways',
    )
    df['regimen_tendencia'] = pd.Series(regimen, index=df.index).mask(sin_historia)

    w_rango = cfg.X3_VENTANAS['drawdown'][0]
    max_previo = high.rolling(w_rango).max().shift(1)
    min_previo = low.rolling(w_rango).min().shift(1)
    sin_rango = max_previo.isna() | min_previo.isna()
    ruptura = np.select([close > max_previo, close < min_previo], [1, -1], default=0).astype(float)
    df['ruptura_rango'] = pd.Series(ruptura, index=df.index).mask(sin_rango)

    hubo_ruptura = df['ruptura_rango'].fillna(0) != 0
    episodio_rango = hubo_ruptura.cumsum()
    df['duracion_en_rango'] = close.groupby(episodio_rango).cumcount()

    return df


tablas = {valor: _variables_tendencia_regimen(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    conteo_regimen = tabla['regimen_tendencia'].value_counts(dropna=True).to_dict()
    print(f"{valor}: {tabla.shape[1]} columnas — regimen_tendencia {conteo_regimen}, "
          f"ruptura_rango no nulos {int(tabla['ruptura_rango'].notna().sum())}")

BTCUSD: 72 columnas — regimen_tendencia {'bear': 14078, 'bull': 13475, 'sideways': 12681}, ruptura_rango no nulos 40263


## 9. Catálogo de documentación por variable (Q1-9)

Tabla con una fila por columna de la tabla final — incluidas las heredadas de
precio/X3/X2, no solo las agregadas en este notebook — con fuente,
significado, unidad, fórmula, si aplica forward-fill, granularidad
esperada/observada, primera/última observación, cobertura, % nulos, edad de
la información propagada (cuando aplica) y limitaciones de interpretación.
Referencia rápida para `X5_Q2.ipynb`/`X5_Q3.ipynb` sin tener que releer cada
celda de este notebook.

Metadata estática (fuente/significado/unidad/fórmula/forward_fill/
limitaciones, documentada a mano por variable) + estadísticas calculadas
sobre los datos reales de cada activo (cobertura/nulos/fechas/edad) — lo que
se puede medir no se documenta a mano.

In [11]:
METADATA_VARIABLES = {}


def _registrar(col, fuente, significado, unidad, formula, forward_fill=False, limitaciones=''):
    METADATA_VARIABLES[col] = {
        'fuente': fuente,
        'significado': significado,
        'unidad': unidad,
        'formula': formula,
        'forward_fill': forward_fill,
        'limitaciones': limitaciones,
    }


# --- Precio (MT5, Data/{valor}.csv) ---
_registrar('DateTime', 'Precio (MT5)', 'Timestamp de la vela H1', 'fecha/hora', 'dato crudo del CSV')
_registrar('Open', 'Precio (MT5)', 'Precio de apertura de la vela', 'USD', 'dato crudo del CSV')
_registrar('High', 'Precio (MT5)', 'Precio máximo de la vela', 'USD', 'dato crudo del CSV')
_registrar('Low', 'Precio (MT5)', 'Precio mínimo de la vela', 'USD', 'dato crudo del CSV')
_registrar('Close', 'Precio (MT5)', 'Precio de cierre de la vela', 'USD', 'dato crudo del CSV')
_registrar('Tick_Volume', 'Precio (MT5)', 'Cantidad de ticks ocurridos en la vela — proxy de actividad', 'conteo', 'dato crudo del CSV',
           limitaciones='Real_Volume no lo reporta el bróker (ver abajo) — Tick_Volume es el proxy de volumen usado en todo el proyecto, no solo aquí.')
_registrar('Spread', 'Precio (MT5)', 'Spread cotizado por el bróker al cierre de la vela', 'puntos MT5 (entero)', 'dato crudo del CSV',
           limitaciones='En puntos del bróker, no en USD. Convertir a precio requiere symbol_info(valor).point, no disponible en Mac ni persistido en los CSV — por eso spread_relativo (Sección 5) normaliza contra su propio promedio móvil en vez de contra Close.')
_registrar('Real_Volume', 'Precio (MT5)', 'Volumen real reportado por el bróker', 'USD/unidades', 'dato crudo del CSV',
           limitaciones='Siempre 0 para este símbolo — el bróker no lo reporta. No usar como variable; Tick_Volume es el proxy válido.')

# --- X3 (técnicos, X3_technical_features.py — reusados tal cual, sin reimplementar) ---
for w in cfg.X3_VENTANAS['sma']:
    _registrar(f'x3_sma_dist_{w}', 'X3 (técnico)', f'Distancia de Close a la SMA de {w} períodos, normalizada por Close',
               'ratio', f'(Close - SMA_{w}) / Close',
               limitaciones=f'NaN en las primeras {w - 1} velas (warm-up de la media móvil).')
for w in cfg.X3_VENTANAS['ema']:
    _registrar(f'x3_ema_dist_{w}', 'X3 (técnico)', f'Distancia de Close a la EMA de {w} períodos, normalizada por Close',
               'ratio', f'(Close - EMA_{w}) / Close')
_registrar('x3_rsi_14', 'X3 (técnico)', 'Relative Strength Index (RSI) de Wilder, ventana 14', 'índice 0-100',
           '100 - 100/(1+RS), RS = EMA_Wilder(ganancias,14) / EMA_Wilder(pérdidas,14)',
           limitaciones='NaN en la primera vela (sin delta previo) y si avg_loss=0 durante el warm-up inicial.')
_registrar('x3_macd', 'X3 (técnico)', 'Línea MACD(12,26,9) — diferencia entre EMA rápida y lenta', 'USD', 'EMA_12(Close) - EMA_26(Close)')
_registrar('x3_macd_signal', 'X3 (técnico)', 'Señal del MACD — EMA de 9 períodos de la línea MACD', 'USD', 'EMA_9(MACD)')
_registrar('x3_macd_hist', 'X3 (técnico)', 'Histograma MACD — distancia entre la línea y su señal', 'USD', 'MACD - MACD_signal')
_registrar('x3_atr_14', 'X3 (técnico)', 'Average True Range (ATR) de Wilder, ventana 14', 'USD',
           'EMA_Wilder(TR, 14), TR = max(High-Low, |High-Close_prev|, |Low-Close_prev|)')
_registrar('x3_atr_pct', 'X3 (técnico)', 'ATR relativo al precio de cierre', 'ratio', 'ATR_14 / Close')
_registrar('x3_bb_width', 'X3 (técnico)', 'Ancho de las Bandas de Bollinger(20,2), normalizado por la media móvil', 'ratio',
           '(banda_superior - banda_inferior) / SMA_20, bandas = SMA_20 ± 2·std_20(Close)',
           limitaciones='NaN en las primeras 19 velas (warm-up).')
_registrar('x3_bb_pos', 'X3 (técnico)', 'Posición de Close dentro de las Bandas de Bollinger(20,2)', 'ratio (≈0-1, puede exceder si rompe banda)',
           '(Close - banda_inferior) / (banda_superior - banda_inferior)',
           limitaciones='NaN en las primeras 19 velas; puede salir de [0,1] si Close rompe la banda superior/inferior.')
for w in cfg.X3_VENTANAS['roc']:
    _registrar(f'x3_roc_{w}', 'X3 (técnico)', f'Rate of Change — retorno simple en los últimos {w} períodos', 'ratio',
               f'(Close - Close.shift({w})) / Close.shift({w})')
_registrar('x3_vol_24h', 'X3 (técnico)', 'Volatilidad realizada anualizada de retornos log, ventana 24h', 'ratio anualizado',
           'std_24(retorno_log) · √8760')
_registrar('x3_vol_7d', 'X3 (técnico)', 'Volatilidad realizada anualizada de retornos log, ventana 7 días (168h)', 'ratio anualizado',
           'std_168(retorno_log) · √8760',
           limitaciones='NaN en las primeras 167 velas (warm-up); consumida por cambio_volatilidad (Sección 4).')
_registrar('x3_drawdown_20', 'X3 (técnico)', 'Drawdown contra el máximo rodante de 20 velas (corto plazo)', 'ratio (≤0)',
           '(Close - max_20(Close)) / max_20(Close)',
           limitaciones='Resetea contra un máximo de solo 20 velas — no confundir con drawdown_historico/drawdown_Xd (Sección 3), que usan el máximo histórico o ventanas más largas.')
_registrar('x3_drawdown_50', 'X3 (técnico)', 'Drawdown contra el máximo rodante de 50 velas (mediano plazo)', 'ratio (≤0)',
           '(Close - max_50(Close)) / max_50(Close)')
_registrar('x3_trend_slope_20', 'X3 (técnico)', 'Pendiente OLS de Close en 20 velas, normalizada por Close', 'ratio',
           'pendiente_OLS(Close, 20) · 20 / Close')
_registrar('x3_trend_slope_50', 'X3 (técnico)', 'Pendiente OLS de Close en 50 velas, normalizada por Close', 'ratio',
           'pendiente_OLS(Close, 50) · 50 / Close',
           limitaciones='Usada por regimen_tendencia (Sección 8) para distinguir bull/bear cuando adx_14 ≥ 20.')

# --- X2 (fundamental, merge_asof backward = forward-fill del último score conocido) ---
_LIM_X2 = ('Forward-fill desde el último registro real de x2_history.json. En el snapshot local de Mac, '
           'Data/BTCUSD.csv termina el 2026-06-02 y los registros de x2_history.json para BTCUSD son todos '
           'posteriores (2026-06-12+) → queda 100% NaN para BTCUSD con estos datos (no es un bug, ver done.md Q1-10). '
           'Se valida al refrescar Data/ desde Windows con fechas que se solapen con x2_history.json.')
_registrar('x2_score', 'X2 (fundamental)', 'Score fundamental agregado del activo', 'score [0,1]',
           'merge_asof(x2_history, direction="backward")', forward_fill=True, limitaciones=_LIM_X2)
_registrar('x2_score_cross', 'X2 (fundamental)', 'Componente cross-sectional del score fundamental, centrado en 0.5', 'score ~[0,1], centro 0.5',
           'merge_asof(x2_history, direction="backward")', forward_fill=True, limitaciones=_LIM_X2)
_registrar('x2_score_tendencia', 'X2 (fundamental)', 'Componente de tendencia del score fundamental, centrado en 0.5', 'score ~[0,1], centro 0.5',
           'merge_asof(x2_history, direction="backward")', forward_fill=True, limitaciones=_LIM_X2)

# --- Técnicos filtrados/transformados (Sección 2, Q1-3) ---
for w in cfg.X3_VENTANAS['sma']:
    _registrar(f'distancia_sma_{w}', 'Q1 — Técnico transformado', f'Distancia de Close a la SMA de {w}, normalizada por ATR (no por Close)',
               'múltiplos de ATR', f'(Close - x3_sma_{w}) / x3_atr_14')
    _registrar(f'pendiente_sma_{w}', 'Q1 — Técnico transformado', f'Cambio promedio por período de la SMA de {w} en los últimos {w} períodos, normalizado por ATR',
               'múltiplos de ATR por período', f'(x3_sma_{w}.diff({w}) / {w}) / x3_atr_14')
for w in cfg.X3_VENTANAS['ema']:
    _registrar(f'distancia_ema_{w}', 'Q1 — Técnico transformado', f'Distancia de Close a la EMA de {w}, normalizada por ATR',
               'múltiplos de ATR', f'(Close - x3_ema_{w}) / x3_atr_14')
    _registrar(f'pendiente_ema_{w}', 'Q1 — Técnico transformado', f'Cambio promedio por período de la EMA de {w} en los últimos {w} períodos, normalizado por ATR',
               'múltiplos de ATR por período', f'(x3_ema_{w}.diff({w}) / {w}) / x3_atr_14')

# --- Drawdown del activo (Sección 3, Q1-4) ---
_registrar('drawdown_historico', 'Q1 — Drawdown', 'Drawdown contra el máximo histórico acumulado de Close', 'ratio (≤0)',
           '(Close - cummax(Close)) / cummax(Close)')
_registrar('duracion_drawdown', 'Q1 — Drawdown', 'Velas transcurridas desde el último máximo histórico', 'períodos (horas, H1)',
           'cumcount() dentro del episodio definido por nuevos máximos históricos')
_registrar('recuperacion_desde_minimo', 'Q1 — Drawdown', 'Recuperación de Close desde el mínimo del episodio de drawdown actual', 'ratio (≥0)',
           '(Close - min_episodio) / min_episodio')
_registrar('velocidad_caida', 'Q1 — Drawdown', 'Profundidad promedio del drawdown por período transcurrido', 'ratio por período',
           'drawdown_historico / duracion_drawdown',
           limitaciones='NaN en la vela exacta de un nuevo máximo histórico (duracion_drawdown=0, división por cero evitada con replace(0,NaN)).')
for dias, horas in [(7, 168), (30, 720), (90, 2160), (365, 8760)]:
    _registrar(f'drawdown_{dias}d', 'Q1 — Drawdown', f'Drawdown contra el máximo rodante de {dias} días ({horas}h)', 'ratio (≤0)',
               f'(Close - max_{horas}(Close)) / max_{horas}(Close)',
               limitaciones='min_periods=1 → definido desde la primera vela (a diferencia de x3_drawdown_20/50, que exigen la ventana completa).')

# --- Volatilidad (Sección 4, Q1-5) ---
_registrar('cambio_volatilidad', 'Q1 — Volatilidad', 'Salto de volatilidad reciente (24h) sobre la línea base de 7 días', 'ratio',
           'x3_vol_24h / x3_vol_7d', limitaciones='NaN mientras x3_vol_7d no tenga historia suficiente (warm-up de 167 velas).')
_registrar('vol_alcista_30d', 'Q1 — Volatilidad', 'Volatilidad anualizada de solo los retornos log positivos, ventana 30d', 'ratio anualizado',
           'std_30d(retorno_log | retorno_log>0) · √8760')
_registrar('vol_bajista_30d', 'Q1 — Volatilidad', 'Volatilidad anualizada de solo los retornos log negativos, ventana 30d', 'ratio anualizado',
           'std_30d(retorno_log | retorno_log<0) · √8760')
_registrar('frecuencia_extremos_30d', 'Q1 — Volatilidad', 'Fracción de horas, en 30d, con retorno log atípico (>2 std expandida)', 'ratio [0,1]',
           'rolling_30d(mean(|retorno_log| > 2·expanding_std(retorno_log)))',
           limitaciones='El umbral usa expanding() (no rolling) para no mirar información futura al definirlo.')

# --- Liquidez (Sección 5, Q1-8) ---
_registrar('spread_relativo', 'Q1 — Liquidez', 'Spread cotizado vs. su propio promedio móvil de 30d', 'ratio (centro ~1)',
           'Spread / media_móvil_30d(Spread)',
           limitaciones='Normalizado contra su propio histórico, no contra Close — ver limitación de Spread arriba (sin factor de conversión a precio).')
_registrar('tick_volume_z', 'Q1 — Liquidez', 'Z-score de Tick_Volume vs. media/std móviles de 30d', 'desviaciones estándar',
           '(Tick_Volume - media_30d) / std_30d')
_registrar('volumen_relativo', 'Q1 — Liquidez', 'Volumen relativo (RVOL) de Tick_Volume vs. su propio promedio móvil de 30d', 'ratio (centro ~1)',
           'Tick_Volume / media_móvil_30d(Tick_Volume)')

# --- Fundamentales de estado (Sección 6, Q1-10) ---
_LIM_FUND_DERIVADA = ('Depende de x2_score, que con los datos actuales de Mac queda 100% NaN para BTCUSD '
                      '(ver limitación de x2_score arriba) → esta columna hereda ese NaN o queda en 0 actualizaciones.')
_registrar('fundamental_cambio', 'Q1 — Fundamental (estado)', 'Cambio de x2_score respecto a la vela anterior', 'score (delta)',
           'x2_score.diff()', limitaciones=_LIM_FUND_DERIVADA)
_registrar('fundamental_direccion_cambio', 'Q1 — Fundamental (estado)', 'Signo del cambio de x2_score', 'categórica (-1/0/1)',
           'sign(fundamental_cambio)', limitaciones=_LIM_FUND_DERIVADA)
_registrar('fundamental_magnitud_cambio', 'Q1 — Fundamental (estado)', 'Magnitud absoluta del cambio de x2_score', 'score (delta absoluto)',
           '|fundamental_cambio|', limitaciones=_LIM_FUND_DERIVADA)
_registrar('fundamental_edad', 'Q1 — Fundamental (estado)', 'Velas H1 desde la última actualización real de x2_score', 'períodos (horas, H1)',
           'cumcount() dentro del episodio definido por actualizaciones reales (fundamental_cambio != 0)',
           limitaciones='Con 0 actualizaciones reales en el historial, crece sin reiniciarse nunca (máximo = todo el historial disponible).')
_registrar('fundamental_frecuencia_efectiva_90d', 'Q1 — Fundamental (estado)', 'Fracción de velas, en 90d, que son una actualización real de x2_score', 'ratio [0,1]',
           'rolling_90d(mean(fundamental_cambio.fillna(0) != 0))', limitaciones=_LIM_FUND_DERIVADA)
_registrar('fundamental_estabilidad_90d', 'Q1 — Fundamental (estado)', 'Desviación estándar rodante de x2_score, ventana 90d', 'score (std)',
           'std_90d(x2_score)', limitaciones=_LIM_FUND_DERIVADA)
_registrar('fundamental_contradiccion', 'Q1 — Fundamental (estado)', 'Consistencia entre los componentes cross y tendencia del score (ambos centrados en 0.5)', 'ratio [-0.25,0.25]',
           '(x2_score_cross - 0.5) * (x2_score_tendencia - 0.5)',
           limitaciones=_LIM_FUND_DERIVADA + ' Negativo=contradicción, positivo=consistencia, 0=al menos un componente neutro.')

# --- Ruido (Sección 7, Q1-6) ---
_registrar('ruido_cambios_signo_30d', 'Q1 — Ruido', 'Fracción de velas, en 30d, con cambio de signo del retorno log vs. la anterior', 'ratio [0,1]',
           'rolling_30d(mean(sign(retorno_log) != sign(retorno_log).shift(1)))')
_registrar('ruido_ratio_movimiento_30d', 'Q1 — Ruido', 'Recorrido total vs. desplazamiento neto en 30d (inverso del Efficiency Ratio de Kaufman)', 'ratio (≥1 típico)',
           'suma_30d(|retorno_log|) / |retorno_log acumulado en 30d|',
           limitaciones='Indefinido (NaN) cuando el retorno acumulado en la ventana es ~0 (división por un valor cercano a cero).')
_registrar('ruido_mecha_30d', 'Q1 — Ruido', 'Promedio rodante de la fracción de cada vela que fue mecha (no cuerpo), 30d', 'ratio [0,1]',
           'rolling_30d(mean((High-Low-|Close-Open|)/(High-Low)))',
           limitaciones='Mismo concepto que la fuerza del rechazo de X0 (ver CLAUDE.md), aquí agregado en ventana en vez de por vela.')
_registrar('ruido_atr_sin_direccion_30d', 'Q1 — Ruido', 'ATR promedio en 30d relativo al desplazamiento neto en la misma ventana', 'ratio (≥0)',
           'media_30d(x3_atr_pct) / |retorno_log acumulado en 30d|',
           limitaciones='Misma división sensible a retorno acumulado ~0 que ruido_ratio_movimiento_30d.')

# --- Tendencia / régimen (Sección 8, Q1-7) ---
_registrar('adx_14', 'Q1 — Tendencia/régimen', 'Average Directional Index (ADX) de Wilder, ventana 14 — fuerza de tendencia sin dirección', 'índice 0-100',
           '_wilder_ema(DX,14), DX=100·|+DI−-DI|/(+DI+-DI), +DI/-DI desde +DM/-DM suavizados con _wilder_ema',
           limitaciones='<20 (convención de Wilder) se interpreta como mercado lateral en regimen_tendencia, sea cual sea el signo de la pendiente.')
_registrar('regimen_tendencia', 'Q1 — Tendencia/régimen', 'Categórica bull/bear/sideways del régimen de tendencia', 'categórica',
           "'sideways' si adx_14<20; si no, 'bull'/'bear' según signo de x3_trend_slope_50",
           limitaciones='NaN durante el warm-up de adx_14/x3_trend_slope_50.')
_registrar('ruptura_rango', 'Q1 — Tendencia/régimen', 'Ruptura estilo Donchian contra el rango de las 20 velas previas', 'categórica (-1/0/1)',
           '+1 si Close>max_20_previo, -1 si Close<min_20_previo, 0 si está dentro del rango',
           limitaciones='NaN en el warm-up (primeras 20 velas sin rango previo completo).')
_registrar('duracion_en_rango', 'Q1 — Tendencia/régimen', 'Períodos transcurridos desde la última ruptura de rango', 'períodos (horas, H1)',
           'cumcount() dentro del episodio definido por ruptura_rango != 0')

print(f"Metadata registrada para {len(METADATA_VARIABLES)} variables")

Metadata registrada para 72 variables


In [12]:
_COLS_LIGADAS_X2 = {
    'x2_score', 'x2_score_cross', 'x2_score_tendencia',
    'fundamental_cambio', 'fundamental_direccion_cambio', 'fundamental_magnitud_cambio',
    'fundamental_frecuencia_efectiva_90d', 'fundamental_estabilidad_90d', 'fundamental_contradiccion',
}


def _granularidad_observada(df, col):
    if col not in _COLS_LIGADAS_X2:
        return 'H1 (una observación nueva por vela, igual que el resto de la tabla)'
    n_actualizaciones = int((df['x2_score'].diff().fillna(0) != 0).sum())
    if n_actualizaciones == 0:
        return '0 actualizaciones reales de x2_score en este snapshot (ver limitaciones)'
    return f'{n_actualizaciones} actualizaciones reales de x2_score en el historial'


def _catalogo_variables(valor, df):
    faltantes = set(df.columns) - set(METADATA_VARIABLES)
    sobrantes = set(METADATA_VARIABLES) - set(df.columns)
    if faltantes or sobrantes:
        raise ValueError(f"{valor}: metadata desalineada — faltan {sorted(faltantes)}, sobran {sorted(sobrantes)}")

    filas = []
    for col in df.columns:
        meta = METADATA_VARIABLES[col]
        serie = df[col]
        no_nulos = serie.dropna()
        cobertura_pct = round(100 * serie.notna().mean(), 2)
        if col == 'DateTime' or no_nulos.empty:
            primera_obs = ultima_obs = pd.NaT
        else:
            primera_obs = df.loc[no_nulos.index[0], 'DateTime']
            ultima_obs = df.loc[no_nulos.index[-1], 'DateTime']
        filas.append({
            'variable': col,
            'fuente': meta['fuente'],
            'significado': meta['significado'],
            'unidad': meta['unidad'],
            'formula': meta['formula'],
            'forward_fill': meta['forward_fill'],
            'granularidad_esperada': ('diaria o más esparsa en origen (X2), propagada a H1 por forward-fill'
                                       if meta['forward_fill'] else 'H1'),
            'granularidad_observada': _granularidad_observada(df, col),
            'primera_observacion': primera_obs,
            'ultima_observacion': ultima_obs,
            'cobertura_pct': cobertura_pct,
            'pct_nulos': round(100 - cobertura_pct, 2),
            'edad_informacion_propagada': 'ver fundamental_edad' if meta['forward_fill'] else 'N/A',
            'limitaciones': meta['limitaciones'] or 'Ninguna relevante adicional a la de su fuente.',
        })
    return pd.DataFrame(filas)


catalogos = {valor: _catalogo_variables(valor, tabla) for valor, tabla in tablas.items()}

for valor, catalogo in catalogos.items():
    path_out = cfg.CARPETA_X5_ALT / f'{valor}_catalogo_variables.csv'
    catalogo.to_csv(path_out, index=False)
    print(f"{valor}: catálogo de {len(catalogo)} variables guardado en {path_out}")

catalogos['BTCUSD'][['variable', 'fuente', 'unidad', 'cobertura_pct', 'granularidad_observada']].head(15)

BTCUSD: catálogo de 72 variables guardado en /Users/macasaez/Desktop/claude_projects_v2/Alginvesting/resources/x5_alt/BTCUSD_catalogo_variables.csv


,variable,fuente,unidad,cobertura_pct,granularidad_observada
0,DateTime,Precio (MT5),fecha/hora,100.00,"H1 (una observación nueva por vela, igual que ..."
1,Open,Precio (MT5),USD,100.00,"H1 (una observación nueva por vela, igual que ..."
2,High,Precio (MT5),USD,100.00,"H1 (una observación nueva por vela, igual que ..."
3,Low,Precio (MT5),USD,100.00,"H1 (una observación nueva por vela, igual que ..."
4,Close,Precio (MT5),USD,100.00,"H1 (una observación nueva por vela, igual que ..."
5,Tick_Volume,Precio (MT5),conteo,100.00,"H1 (una observación nueva por vela, igual que ..."
6,Spread,Precio (MT5),puntos MT5 (entero),100.00,"H1 (una observación nueva por vela, igual que ..."
7,Real_Volume,Precio (MT5),USD/unidades,100.00,"H1 (una observación nueva por vela, igual que ..."
8,x3_sma_dist_20,X3 (técnico),ratio,99.95,"H1 (una observación nueva por vela, igual que ..."
9,x3_sma_dist_50,X3 (técnico),ratio,99.88,"H1 (una observación nueva por vela, igual que ..."


## 10. Guardar tabla extendida (Q1-2)

A `resources/x5_alt/{valor}_tabla_estado_v2.csv` — separada de
`{valor}_tabla_maestra.csv` de v1 (`X5_P1.ipynb`) para no romper nada que ya
consuma el archivo original. Esta tabla (precio + X3 + X2 + variables de
estado de las Secciones 2-8) es la que consume `X5_Q2.ipynb`, no la de v1.

In [13]:
cfg.CARPETA_X5_ALT.mkdir(parents=True, exist_ok=True)  # resources/x5_alt/ no existe en un checkout nuevo

for valor, tabla in tablas.items():
    path_out = cfg.CARPETA_X5_ALT / f'{valor}_tabla_estado_v2.csv'
    tabla.to_csv(path_out, index=False)
    print(f"{valor}: tabla extendida ({tabla.shape[0]} filas x {tabla.shape[1]} columnas) guardada en {path_out}")

BTCUSD: tabla extendida (40283 filas x 72 columnas) guardada en /Users/macasaez/Desktop/claude_projects_v2/Alginvesting/resources/x5_alt/BTCUSD_tabla_estado_v2.csv


## 11. Exportar resultados a zip (Windows → Mac)

Este notebook corre en Windows con `Data/`/`resources/x2/`/`resources/x3/`
actualizados (no disponibles así en Mac, ver `CLAUDE.md`). Para pasarle el
resultado a Claude sin subir esos archivos a git, la última celda empaqueta
en un único `.zip` todo lo generado en las Secciones 9-10 (tabla extendida +
catálogo de variables, por cada activo de `valores`) usando `exportar_zip`
(`scripts/x5_alt_export.py`, reutilizada tal cual por `X5_Q2.ipynb`/
`X5_Q3.ipynb` — ver `docs/plans/X5_alternativo.md` sección 10.2).

In [14]:
from x5_alt_export import exportar_zip  # noqa: E402

_archivos = []
for valor in valores:
    _archivos.append(cfg.CARPETA_X5_ALT / f'{valor}_tabla_estado_v2.csv')
    _archivos.append(cfg.CARPETA_X5_ALT / f'{valor}_catalogo_variables.csv')

exportar_zip('X5_Q1', _archivos)

X5_Q1: zip con 2 archivo(s) guardado en /Users/macasaez/Desktop/claude_projects_v2/Alginvesting/resources/x5_alt/X5_Q1_20261010_165556.zip


PosixPath('/Users/macasaez/Desktop/claude_projects_v2/Alginvesting/resources/x5_alt/X5_Q1_20261010_165556.zip')